In [ ]:
import warnings
warnings.filterwarnings("ignore")

# numpy 2.x compat shim for the (older) backtrader_plotting / bokeh 2.x stack
import numpy as np
for _o, _n in [("bool8", "bool_"), ("float_", "float64"), ("complex_", "complex128"),
               ("unicode_", "str_"), ("object0", "object_"), ("int0", "intp"), ("uint0", "uintp")]:
    if not hasattr(np, _o):
        setattr(np, _o, getattr(np, _n))

import pandas as pd
import backtrader as bt
from backtrader_plotting import Bokeh
from backtrader_plotting.schemes import Blackly

# clean chart titles: keep just the market name, drop the "SMA(10) @ (...^close)" soup
from backtrader_plotting.bokeh.figure import Figure as _BTPFigure
def _clean_title(self, title):
    title = title.split(" @ ")[0].replace("_", ".").strip()
    if not self.bfigure.title.text:
        self.bfigure.title.text = title
        self.bfigure.title.text_font_size = "12px"
        self.bfigure.title.text_font_style = "bold"
_BTPFigure._figure_append_title = _clean_title

In [ ]:
TITLE = "Claude 4.8 released by May 31?"
df = pd.read_csv("processed/backtest_market.csv", parse_dates=["timestamp"])
o = (df[["timestamp", "price", "usd_amount"]].set_index("timestamp").resample("2min")
     .agg({"price": ["first", "max", "min", "last"], "usd_amount": "sum"}))
o.columns = ["open", "high", "low", "close", "volume"]
o = o.dropna()

In [ ]:
class RSI2(bt.Strategy):
    """Connors-style mean reversion: buy deep oversold dips, sell back into strength."""
    params = (("rp", 2), ("lo", 15), ("exit", 5))
    def __init__(self):
        self.rsi = bt.ind.RSI(self.data.close, period=self.p.rp)
        self.ma = bt.ind.SMA(self.data.close, period=self.p.exit)
    def next(self):
        if not self.position and self.rsi[0] < self.p.lo:
            self.buy(size=self.broker.getcash() * 0.97 / self.data.close[0])
        elif self.position and self.data.close[0] > self.ma[0]:
            self.close()

In [ ]:
cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=o, openinterest=-1), name=TITLE)
cerebro.addstrategy(RSI2)
cerebro.broker.setcash(10000.0)
cerebro.broker.setcommission(commission=0)
print(f"Start: ${cerebro.broker.getvalue():,.2f}")
cerebro.run()
print(f"Final: ${cerebro.broker.getvalue():,.2f}")

In [ ]:
from IPython.display import IFrame, clear_output

b = Bokeh(style="bar", plot_mode="single", scheme=Blackly(),
          output_mode="save", filename="processed/plot.html")
cerebro.plot(b, iplot=False)

clear_output()
IFrame("processed/plot.html", width="100%", height=660)